<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/Filter_Bed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
## Load Google Drive

## Connect VM to User Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/drive


## install dependency, download data and annotation

In [13]:
! apt-get install bedtools
! mkdir results
! mkdir data/
! tar -xvzf /content/drive/My\ Drive/project_notebook/Mustard_Nandan_results/Results.tar.gz -C data/
! mkdir annotation
! wget ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz -P annotation/

DIS3L2_gene_list.slop5000.sorted.bed
all.predictions.bedGraph.gz
hotspots_per_threshold/hotspots.unstranded.threshold.0.6.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.65.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.75.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.35.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.4.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.1.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.2.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.85.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.8.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.5.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.9.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.15.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.25.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.95.bed
hotspots_per_threshold/hotspots.unstranded.threshold.0.7.bed
hotspots_per_

## Filter annotation for UTR

In [0]:
! zcat annotation/gencode.v32.primary_assembly.annotation.gtf.gz | awk '$3=="UTR" {print $0}' > annotation/gencode.v32.primary_assembly.annotation.UTR.gtf

## Arbitrary Extend UTR coordinates

In [0]:
def resize_intervals(annotation, extend, strand=False):
  output_name = f'utr_extended_{extend}.gtf'
  with open(annotation, 'r') as f_input, open(output_name, "w") as f_output:
    for index, line in enumerate(f_input):
      columns = line.split('\t')
      #print('after', columns)
      columns[3] = str(int(columns[3]) - extend)
      columns[4] = str(int(columns[4]) + extend)
      #print('extended', columns)
      
      new_line = '\t'.join(columns)
      f_output.write(new_line)

      #print(new_line)
      # if index==10:
      #    break
  return output_name



## Intersect with BedTools

In [0]:
import subprocess


def run_bedtools(candidate_file, annotation, outdir):
  candidate_file_name = os.path.basename(candidate_file).replace('.bed', '.filtered.bed')
  outname = f'{outdir}/{candidate_file_name}'
  cmd = f'bedtools intersect -a {candidate_file} -b {annotation} -u > {outname}'
  try:
    subprocess.run(cmd, shell=True)
  except:
    return "error"




## run code

In [95]:
import os


annotation_path = "annotation/gencode.v32.primary_assembly.annotation.UTR.gtf"
bed_dir_path = "data/hotspots_per_threshold/"
output_dir_path = "results/"
extend_size = 1000
strand=False

# extend UTR boundaries
resized_file = resize_intervals(annotation_path, extend_size, strand)
# filter candidates
for bed_file in os.listdir(bed_dir_path):
  candidates_bed_path = os.path.join(bed_dir_path, bed_file)
  print(candidates_bed_path)
  run_bedtools(candidates_bed, resized_file, output_dir_path)

data/hotspots_per_threshold/hotspots.unstranded.threshold.0.25.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.7.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.8.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.75.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.6.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.45.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.85.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.2.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.3.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.1.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.4.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.9.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.55.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.95.bed
data/hotspots_per_threshold/hotspots.unstranded.threshold.0.5.bed
data